## Трансформации

In [ ]:
from torchvision import transforms

# Статистики ImageNet для нормализации
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# === TRAIN: аугментации и подготовка
train_transforms = transforms.Compose([    
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.9, 1.1)),     # сохраняет мелкие детали (крошки, пятна)
    transforms.RandomHorizontalFlip(p=0.5),      # ломаем ориентацию узора
    transforms.RandomRotation(degrees=30),        # поворот    
    transforms.ColorJitter(     # Освещение (блики)
        brightness=0.3,   # ±30% яркости
        contrast=0.3,     # ±30% контраста
        saturation=0.2,   # ±20% насыщенности
        hue=0.05          # лёгкий сдвиг оттенка
    ),        
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),   #Blur (заблюренный фон)    
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), # нормализация
])

# === VAL/TEST: только подготовка (без аугментаций)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## Dataset для обучения

In [ ]:
import os
from pathlib import Path
from typing import Tuple, List

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from PIL import Image

# Создаем тренировочный датасет
train_dataset = datasets.ImageFolder(root='data/train', transform=train_transforms)

# Смотрим, какие классы он нашел и какие индексы им присвоил
print("Классы:", train_dataset.classes)       # Ожидаем: ['cleaned', 'dirty']
print("Маппинг классов:", train_dataset.class_to_idx) # Ожидаем: {'cleaned': 0, 'dirty': 1}

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)